In [1]:
import os
import sys

PROJECT_DIR = (
    "/path/to/"
    "Maya Archarological Sites/Model Training/"
    "swin3d_project"
)

os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(
        0,
        PROJECT_DIR,
    )

print("Working directory:", os.getcwd())

Working directory: /path/to/Maya Archarological Sites/Model Training/swin3d_project


In [2]:
from pathlib import Path

import torch

from models.swin3d_model import Model
from datasets.belize_laz_dataset import create_dataloaders
from evaluation.evaluate_avd import evaluate_best_model_avd

/path/to/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


In [3]:
_, _, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=1,
    voxel_max=65_000,
    num_workers=0,
)

print("Test batches:", len(test_loader))

Test batches: 80


In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = Model(
    num_classes=3,
    in_channels=4,
).to(device)

print("Device:", device)

Device: cuda


In [5]:
BEST_CHECKPOINT = Path(
    "/path/to/project_storage/"
    "swin3d_checkpoints/"
    "dice_vicinity/"
    "swin3d_dice_vicinity_best_02.pth"
)

AVD_RESULTS_DIR = Path(
    "/path/to/project_storage/"
    "swin3d_results/generalized_dice/dice_vicinity"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint:", BEST_CHECKPOINT)
print("Output:", AVD_RESULTS_DIR)

Checkpoint exists: True
Checkpoint: /path/to/project_storage/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_best_02.pth
Output: /path/to/project_storage/swin3d_results/generalized_dice/dice_vicinity


In [6]:
sample = test_loader.dataset[0]

print("coord:", sample["coord"].shape)
print("metric_coord:", sample["metric_coord"].shape)
print("feat:", sample["feat"].shape)
print("segment:", sample["segment"].shape)

assert sample["coord"].shape[0] == sample["metric_coord"].shape[0]
assert sample["coord"].shape[0] == sample["feat"].shape[0]
assert sample["coord"].shape[0] == sample["segment"].shape[0]

coord: torch.Size([65000, 3])
metric_coord: torch.Size([65000, 3])
feat: torch.Size([65000, 1])
segment: torch.Size([65000])


avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR,
    device=device,

    cell_size_m=1.0,
    radii_m=(2.0, 5.0, 10.0),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

In [7]:
from torch.utils.data import DataLoader, Subset

small_test_dataset = Subset(
    test_loader.dataset,
    range(min(80, len(test_loader.dataset))),
)

small_test_loader = DataLoader(
    small_test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=test_loader.collate_fn,
)

print("Tiles in quick test:", len(small_test_dataset))

Tiles in quick test: 80


In [8]:
avd_output = evaluate_best_model_avd(
    model=model,
    test_loader=small_test_loader,
    checkpoint_path=BEST_CHECKPOINT,
    output_dir=AVD_RESULTS_DIR / "quick_test",
    device=device,

    cell_size_m=1.0,
    radii_m=(5.0,),
    min_points_per_cell=1,
    min_evidence_cells=3,
    beta=2.0,

    print_progress=True,
    save_csv=True,
)

/path/to/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/path/to/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Evaluated AVD batch 1/80
Evaluated AVD batch 5/80
Evaluated AVD batch 10/80
Evaluated AVD batch 15/80
Evaluated AVD batch 20/80
Evaluated AVD batch 25/80
Evaluated AVD batch 30/80
Evaluated AVD batch 35/80
Evaluated AVD batch 40/80
Evaluated AVD batch 45/80
Evaluated AVD batch 50/80
Evaluated AVD batch 55/80
Evaluated AVD batch 60/80
Evaluated AVD batch 65/80
Evaluated AVD batch 70/80
Evaluated AVD batch 75/80
Evaluated AVD batch 80/80

AVD metric: cell=1 m, minimum evidence=3 cells, F_beta beta=2
class | radius_m | precision | recall | AVD-F_beta | near/total predicted cells | detected/total features
structure | 5 | 0.3552 | 0.2738 | 0.2870 | 1425/4012 | 23/84
plazuela | 5 | 0.5481 | 0.9670 | 0.8388 | 46388/84637 | 88/91
archaeological_discovery | 5 | 0.5663 | 0.9778 | 0.8537 | 49428/87289 | 88/90
macro AVD-F_beta @ 5 m = 0.6598

Checkpoint epoch: 55


In [9]:
avd_results_df = avd_output[
    "results_df"
]

display(avd_results_df)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,23,0.273810,0.355184,0.286958,4012,1425,2587
1,plazuela,5.0,1.0,1,3,2.0,91,88,0.967033,0.548082,0.838798,84637,46388,38249
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,88,0.977778,0.566257,0.853695,87289,49428,37861


In [10]:
avd_macro_df = avd_output[
    "macro_df"
]

display(avd_macro_df)

,Mode / Class,Radius (m),Structure AVD-F2,Plazuela AVD-F2,Macro AVD-F2
0,archaeological_macro,5.0,0.286958,0.838798,0.562878


In [11]:
display(
    avd_results_df[
        avd_results_df["Radius (m)"] == 5.0
    ]
)

,Mode / Class,Radius (m),Grid Cell Size (m),Minimum Points Per Cell,Minimum Evidence Cells,Beta,GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,structure,5.0,1.0,1,3,2.0,84,23,0.273810,0.355184,0.286958,4012,1425,2587
1,plazuela,5.0,1.0,1,3,2.0,91,88,0.967033,0.548082,0.838798,84637,46388,38249
2,archaeological_discovery,5.0,1.0,1,3,2.0,90,88,0.977778,0.566257,0.853695,87289,49428,37861


# Fast AVD Evaluation with Cached Test Predictions

This version runs Swin3D inference **only once per checkpoint** and saves each test tile as a compressed `.npz` file.

Future runs can skip model inference entirely and go directly to:

**cached predictions → spatial filter → AVD before/after**

The cache stores:

- metric coordinates
- ground-truth labels
- raw model predictions
- original tile path


In [12]:
# ============================================================
# Imports for cached evaluation
# ============================================================

from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch

from visualization.vis import spatial_filter_predictions

from archaeological_vicinity_metric import (
    AVDConfig,
    ArchaeologicalVicinityMetric,
    ClassSpec,
)


In [13]:
# ============================================================
# Cache + filter + AVD settings
# ============================================================

CACHE_DIR = Path(
    "/path/to/project_storage/"
    "swin3d_results/"
    "prediction_cache/"
    f"{BEST_CHECKPOINT.stem}"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FILTER_SETTINGS = {
    "cell_size": 1.0,
    "structure_min_cells": 8,
    "plazuela_min_cells": 15,
    "opening_iterations": 1,
    "closing_iterations": 0,
}

AVD_CELL_SIZE = 1.0
AVD_RADII = (5.0,)
AVD_MIN_POINTS_PER_CELL = 1
AVD_MIN_EVIDENCE_CELLS = 3
AVD_BETA = 2.0

print("Cache directory:")
print(CACHE_DIR)

print()
print("Filter settings:")
print(FILTER_SETTINGS)


Cache directory:
/path/to/project_storage/swin3d_results/prediction_cache/swin3d_dice_vicinity_best_02

Filter settings:
{'cell_size': 1.0, 'structure_min_cells': 8, 'plazuela_min_cells': 15, 'opening_iterations': 1, 'closing_iterations': 0}


In [14]:
# ============================================================
# AVD helper functions
#
# Project labels:
# 0 = Ground
# 1 = Structure
# 2 = Plazuela
# ============================================================

def make_avd_metric():
    return ArchaeologicalVicinityMetric(
        class_specs=[
            ClassSpec(
                "structure",
                predicted_labels=1,
                target_labels=1,
            ),

            ClassSpec(
                "plazuela",
                predicted_labels=2,
                target_labels=2,
            ),

            ClassSpec(
                "archaeological_discovery",
                predicted_labels=(1, 2),
                target_labels=(1, 2),
            ),
        ],

        config=AVDConfig(
            cell_size_m=AVD_CELL_SIZE,
            radii_m=AVD_RADII,
            min_points_per_cell=(
                AVD_MIN_POINTS_PER_CELL
            ),
            min_evidence_cells=(
                AVD_MIN_EVIDENCE_CELLS
            ),
            beta=AVD_BETA,
            ignore_label=-100,
        ),
    )


def result_to_dataframe(
    result,
    version,
):
    rows = []

    for class_name in [
        "structure",
        "plazuela",
        "archaeological_discovery",
    ]:

        metrics = (
            result["per_class"]
            [class_name]
            [5.0]
        )

        predicted_cells = int(
            metrics[
                "predicted_cells"
            ]
        )

        useful_cells = int(
            metrics[
                "predicted_cells_near_gt"
            ]
        )

        rows.append(
            {
                "Version": version,
                "Mode / Class": class_name,
                "Radius (m)": 5.0,

                "GT Features": int(
                    metrics[
                        "gt_features"
                    ]
                ),

                "Detected Features": int(
                    metrics[
                        "detected_gt_features"
                    ]
                ),

                "Vicinity Recall": float(
                    metrics[
                        "feature_recall"
                    ]
                ),

                "Vicinity Precision": float(
                    metrics[
                        "vicinity_precision"
                    ]
                ),

                "AVD-F2": float(
                    metrics[
                        "avd_fbeta"
                    ]
                ),

                "Predicted Cells":
                    predicted_cells,

                "Useful Predicted Cells":
                    useful_cells,

                "False Predicted Cells":
                    predicted_cells
                    -
                    useful_cells,
            }
        )

    return pd.DataFrame(
        rows
    )


In [15]:
# ============================================================
# Load checkpoint
# ============================================================

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print(
    "Checkpoint epoch:",
    checkpoint.get(
        "epoch",
        "unknown",
    )
)


/tmp/ipykernel_67757/3772282829.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Checkpoint epoch: 55


In [16]:
from tqdm import tqdm

In [17]:
# ============================================================
# STEP 1
# Generate cache ONLY if it does not already exist
# ============================================================

existing_cache_files = sorted(
    CACHE_DIR.glob(
        "*.npz"
    )
)

if len(existing_cache_files) > 0:

    print(
        f"Cache already exists: "
        f"{len(existing_cache_files)} files"
    )

    print(
        "Skipping Swin3D inference."
    )

else:

    print(
        "No cached predictions found."
    )

    print(
        "Running Swin3D inference once..."
    )

    inference_start = (
        time.perf_counter()
    )

    saved_tiles = 0

    with torch.no_grad():

        for batch_index, batch in enumerate(
            tqdm(
                test_loader,
                desc="Generating prediction cache",
                total=len(test_loader),
                unit="batch",
            )
        ):

            (
                coords,
                features,
                offset,
                labels,
                paths,
                metric_coords,
            ) = batch

            coords_gpu = coords.to(
                device,
                non_blocking=True,
            ).float()

            features_gpu = features.to(
                device,
                non_blocking=True,
            ).float()

            offset_gpu = offset.to(
                device,
                non_blocking=True,
            ).long()

            logits = model(
                (
                    coords_gpu,
                    features_gpu,
                    offset_gpu,
                )
            )

            if isinstance(
                logits,
                tuple,
            ):
                logits = logits[0]

            predictions = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
                .astype(
                    np.uint8
                )
            )

            labels_np = (
                labels
                .cpu()
                .numpy()
                .astype(
                    np.uint8
                )
            )

            metric_coords_np = (
                metric_coords
                .cpu()
                .numpy()
            )

            start = 0

            for tile_in_batch, end in enumerate(
                offset.tolist()
            ):

                original_path = str(
                    paths[
                        tile_in_batch
                    ]
                )

                original_name = (
                    Path(
                        original_path
                    ).stem
                )

                save_path = (
                    CACHE_DIR
                    /
                    f"{saved_tiles:04d}_{original_name}.npz"
                )

                np.savez_compressed(
                    save_path,

                    coords=(
                        metric_coords_np[
                            start:end
                        ]
                    ),

                    labels=(
                        labels_np[
                            start:end
                        ]
                    ),

                    predictions=(
                        predictions[
                            start:end
                        ]
                    ),

                    original_path=np.array(
                        original_path
                    ),
                )

                saved_tiles += 1
                start = end

    inference_seconds = (
        time.perf_counter()
        -
        inference_start
    )

    print()
    print(
        f"Saved {saved_tiles} tiles."
    )

    print(
        f"Inference + cache time: "
        f"{inference_seconds:.2f} sec"
    )

    print(
        "Cache directory:"
    )

    print(
        CACHE_DIR
    )


No cached predictions found.
Running Swin3D inference once...


Generating prediction cache: 100%|███████████████████████████████████████████████████████| 80/80 [00:16<00:00,  4.76batch/s]


Saved 80 tiles.
Inference + cache time: 16.81 sec
Cache directory:
/path/to/project_storage/swin3d_results/prediction_cache/swin3d_dice_vicinity_best_02


In [18]:
# ============================================================
# STEP 2
# Load cached predictions
#
# No model inference is performed in this cell.
# ============================================================

load_start = (
    time.perf_counter()
)

cache_files = sorted(
    CACHE_DIR.glob(
        "*.npz"
    )
)

if len(cache_files) == 0:
    raise RuntimeError(
        "No cached prediction files found."
    )

cached_tiles = []

for file_path in cache_files:

    with np.load(
        file_path,
        allow_pickle=False,
    ) as data:

        original_path = str(
            data[
                "original_path"
            ].item()
        )

        cached_tiles.append(
            {
                "coords": (
                    data[
                        "coords"
                    ]
                    .copy()
                ),

                "labels": (
                    data[
                        "labels"
                    ]
                    .copy()
                ),

                "predictions": (
                    data[
                        "predictions"
                    ]
                    .copy()
                ),

                "path": (
                    original_path
                ),

                "cache_path": str(
                    file_path
                ),
            }
        )


load_seconds = (
    time.perf_counter()
    -
    load_start
)

print(
    "Loaded cached tiles:",
    len(cached_tiles),
)

print(
    f"Cache load time: "
    f"{load_seconds:.2f} sec"
)


Loaded cached tiles: 80
Cache load time: 0.58 sec


In [19]:
# ============================================================
# STEP 3
# Apply spatial filter to cached predictions
# ============================================================

filter_start = (
    time.perf_counter()
)

total_structure_removed = 0
total_plazuela_removed = 0

total_archaeology_before = 0
total_archaeology_after = 0


for tile_index, tile in enumerate(
    cached_tiles
):

    (
        filtered_predictions,
        filter_info,
    ) = spatial_filter_predictions(
        coords=tile[
            "coords"
        ],

        predictions=tile[
            "predictions"
        ],

        **FILTER_SETTINGS,

        verbose=False,
    )

    tile[
        "filtered_predictions"
    ] = filtered_predictions

    total_structure_removed += (
        filter_info[1][
            "removed_points"
        ]
    )

    total_plazuela_removed += (
        filter_info[2][
            "removed_points"
        ]
    )

    total_archaeology_before += int(
        (
            tile[
                "predictions"
            ]
            !=
            0
        ).sum()
    )

    total_archaeology_after += int(
        (
            filtered_predictions
            !=
            0
        ).sum()
    )

    if (
        tile_index == 0
        or
        (tile_index + 1) % 10 == 0
        or
        tile_index + 1
        ==
        len(cached_tiles)
    ):
        print(
            f"Filtered tile "
            f"{tile_index + 1}/"
            f"{len(cached_tiles)}"
        )


filter_seconds = (
    time.perf_counter()
    -
    filter_start
)


print()
print("=" * 65)
print("FILTER SUMMARY")
print("=" * 65)

print(
    "Structure points removed:",
    f"{total_structure_removed:,}",
)

print(
    "Plazuela points removed:",
    f"{total_plazuela_removed:,}",
)

print(
    "Archaeological predictions before:",
    f"{total_archaeology_before:,}",
)

print(
    "Archaeological predictions after:",
    f"{total_archaeology_after:,}",
)

print(
    "Total archaeological points removed:",
    f"{total_archaeology_before - total_archaeology_after:,}",
)

print(
    f"Filter runtime: "
    f"{filter_seconds:.2f} sec"
)


Filtered tile 1/80
Filtered tile 10/80
Filtered tile 20/80
Filtered tile 30/80
Filtered tile 40/80
Filtered tile 50/80
Filtered tile 60/80
Filtered tile 70/80
Filtered tile 80/80

FILTER SUMMARY
Structure points removed: 2,334
Plazuela points removed: 56,597
Archaeological predictions before: 426,731
Archaeological predictions after: 367,800
Total archaeological points removed: 58,931
Filter runtime: 0.21 sec


In [20]:
# ============================================================
# STEP 4
# Hard AVD BEFORE and AFTER filtering
#
# Uses only cached data.
# No Swin3D inference.
# ============================================================

avd_before_metric = (
    make_avd_metric()
)

avd_after_metric = (
    make_avd_metric()
)

avd_start = (
    time.perf_counter()
)


for tile_index, tile in enumerate(
    cached_tiles
):

    coords = tile[
        "coords"
    ]

    labels = tile[
        "labels"
    ]

    raw_predictions = tile[
        "predictions"
    ]

    filtered_predictions = tile[
        "filtered_predictions"
    ]


    # BEFORE
    avd_before_metric.update(
        xyz_m=coords,

        predicted_labels=(
            raw_predictions
        ),

        target_labels=labels,
    )


    # AFTER
    avd_after_metric.update(
        xyz_m=coords,

        predicted_labels=(
            filtered_predictions
        ),

        target_labels=labels,
    )


    if (
        tile_index == 0
        or
        (tile_index + 1) % 10 == 0
        or
        tile_index + 1
        ==
        len(cached_tiles)
    ):
        print(
            f"AVD tile "
            f"{tile_index + 1}/"
            f"{len(cached_tiles)}"
        )


before_result = (
    avd_before_metric.compute()
)

after_result = (
    avd_after_metric.compute()
)


avd_seconds = (
    time.perf_counter()
    -
    avd_start
)


print()
print(
    f"AVD runtime: "
    f"{avd_seconds:.2f} sec"
)


AVD tile 1/80
AVD tile 10/80
AVD tile 20/80
AVD tile 30/80
AVD tile 40/80
AVD tile 50/80
AVD tile 60/80
AVD tile 70/80
AVD tile 80/80

AVD runtime: 1.20 sec


In [21]:
# ============================================================
# STEP 5
# Before / After result tables
# ============================================================

before_df = (
    result_to_dataframe(
        before_result,
        "Before filter",
    )
)

after_df = (
    result_to_dataframe(
        after_result,
        "After filter",
    )
)

results_df = pd.concat(
    [
        before_df,
        after_df,
    ],

    ignore_index=True,
)


display(
    results_df.style.format(
        {
            "Radius (m)":
                "{:.1f}",

            "Vicinity Recall":
                "{:.4f}",

            "Vicinity Precision":
                "{:.4f}",

            "AVD-F2":
                "{:.4f}",
        }
    )
)


,Version,Mode / Class,Radius (m),GT Features,Detected Features,Vicinity Recall,Vicinity Precision,AVD-F2,Predicted Cells,Useful Predicted Cells,False Predicted Cells
0,Before filter,structure,5.0,84,23,0.2738,0.3489,0.2861,4044,1411,2633
1,Before filter,plazuela,5.0,91,88,0.9670,0.5475,0.8385,84901,46480,38421
2,Before filter,archaeological_discovery,5.0,90,88,0.9778,0.5667,0.8539,87526,49602,37924
3,After filter,structure,5.0,84,22,0.2619,0.3654,0.2776,3503,1280,2223
4,After filter,plazuela,5.0,91,80,0.8791,0.6951,0.8349,58352,40562,17790
5,After filter,archaeological_discovery,5.0,90,81,0.9000,0.7087,0.8539,61109,43305,17804


In [22]:
# ============================================================
# STEP 6
# Compact comparison
# ============================================================

comparison_rows = []

for class_name in [
    "structure",
    "plazuela",
    "archaeological_discovery",
]:

    before_row = (
        before_df[
            before_df[
                "Mode / Class"
            ]
            ==
            class_name
        ]
        .iloc[0]
    )

    after_row = (
        after_df[
            after_df[
                "Mode / Class"
            ]
            ==
            class_name
        ]
        .iloc[0]
    )


    comparison_rows.append(
        {
            "Mode / Class":
                class_name,

            "Precision Before":
                before_row[
                    "Vicinity Precision"
                ],

            "Precision After":
                after_row[
                    "Vicinity Precision"
                ],

            "Precision Change":
                after_row[
                    "Vicinity Precision"
                ]
                -
                before_row[
                    "Vicinity Precision"
                ],

            "Recall Before":
                before_row[
                    "Vicinity Recall"
                ],

            "Recall After":
                after_row[
                    "Vicinity Recall"
                ],

            "Recall Change":
                after_row[
                    "Vicinity Recall"
                ]
                -
                before_row[
                    "Vicinity Recall"
                ],

            "AVD-F2 Before":
                before_row[
                    "AVD-F2"
                ],

            "AVD-F2 After":
                after_row[
                    "AVD-F2"
                ],

            "AVD-F2 Change":
                after_row[
                    "AVD-F2"
                ]
                -
                before_row[
                    "AVD-F2"
                ],

            "False Cells Before":
                before_row[
                    "False Predicted Cells"
                ],

            "False Cells After":
                after_row[
                    "False Predicted Cells"
                ],
        }
    )


comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
)


display(
    comparison_df.style.format(
        {
            "Precision Before":
                "{:.4f}",

            "Precision After":
                "{:.4f}",

            "Precision Change":
                "{:+.4f}",

            "Recall Before":
                "{:.4f}",

            "Recall After":
                "{:.4f}",

            "Recall Change":
                "{:+.4f}",

            "AVD-F2 Before":
                "{:.4f}",

            "AVD-F2 After":
                "{:.4f}",

            "AVD-F2 Change":
                "{:+.4f}",
        }
    )
)


,Mode / Class,Precision Before,Precision After,Precision Change,Recall Before,Recall After,Recall Change,AVD-F2 Before,AVD-F2 After,AVD-F2 Change,False Cells Before,False Cells After
0,structure,0.3489,0.3654,+0.0165,0.2738,0.2619,-0.0119,0.2861,0.2776,-0.0085,2633,2223
1,plazuela,0.5475,0.6951,+0.1477,0.9670,0.8791,-0.0879,0.8385,0.8349,-0.0036,38421,17790
2,archaeological_discovery,0.5667,0.7087,+0.1419,0.9778,0.9000,-0.0778,0.8539,0.8539,-0.0000,37924,17804


In [23]:
# ============================================================
# STEP 7
# Macro AVD-F2 before / after
# ============================================================

def semantic_macro(
    dataframe
):

    semantic = dataframe[
        dataframe[
            "Mode / Class"
        ].isin(
            [
                "structure",
                "plazuela",
            ]
        )
    ]

    return float(
        semantic[
            "AVD-F2"
        ].mean()
    )


macro_before = semantic_macro(
    before_df
)

macro_after = semantic_macro(
    after_df
)


macro_df = pd.DataFrame(
    [
        {
            "Version":
                "Before filter",

            "Macro AVD-F2":
                macro_before,
        },

        {
            "Version":
                "After filter",

            "Macro AVD-F2":
                macro_after,
        },
    ]
)


display(
    macro_df.style.format(
        {
            "Macro AVD-F2":
                "{:.4f}",
        }
    )
)

print(
    "Macro AVD-F2 change:",
    f"{macro_after - macro_before:+.4f}",
)


,Version,Macro AVD-F2
0,Before filter,0.5623
1,After filter,0.5563


Macro AVD-F2 change: -0.0060


In [24]:
# ============================================================
# OPTIONAL
# Save before / after comparison CSVs
# ============================================================

OUTPUT_COMPARE_DIR = (
    CACHE_DIR
    /
    "avd_comparison"
)

OUTPUT_COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

results_path = (
    OUTPUT_COMPARE_DIR
    /
    "avd_before_after.csv"
)

comparison_path = (
    OUTPUT_COMPARE_DIR
    /
    "avd_comparison.csv"
)

macro_path = (
    OUTPUT_COMPARE_DIR
    /
    "avd_macro_comparison.csv"
)


results_df.to_csv(
    results_path,
    index=False,
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

macro_df.to_csv(
    macro_path,
    index=False,
)


print("Saved:")
print(results_path)
print(comparison_path)
print(macro_path)


SyntaxError: positional argument follows keyword argument (276742817.py, line 18)